# 2026 FIFA World Cup — Data Exploration
Loads all raw CSVs, checks data quality, and visualises key distributions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
RAW = Path("../data/raw")

In [ ]:
fifa   = pd.read_csv(RAW / "fifa_rankings.csv")
elo    = pd.read_csv(RAW / "elo_ratings.csv")
matches = pd.read_csv(RAW / "matches.csv", parse_dates=["date"])
mktval = pd.read_csv(RAW / "market_values.csv")
inj    = pd.read_csv(RAW / "injuries.csv")

datasets = {
    "fifa_rankings": fifa,
    "elo_ratings":   elo,
    "matches":       matches,
    "market_values": mktval,
    "injuries":      inj,
}

for name, df in datasets.items():
    date_col = "date" if "date" in df.columns else None
    if date_col:
        print(f"{name}: {len(df):,} rows | date range {df[date_col].min()} → {df[date_col].max()}")
    else:
        print(f"{name}: {len(df):,} rows | no date column")

## FIFA Rankings — Top 20

In [ ]:
print(fifa.sort_values("rank").head(20).to_string(index=False))

## Match Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matches per year
matches["year"] = matches["date"].dt.year
matches.groupby("year").size().plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Matches per Year")
axes[0].set_xlabel("Year")
axes[0].set_ylabel("Number of Matches")
axes[0].tick_params(axis="x", rotation=45)

# Top 10 tournaments
top_tournaments = matches["tournament"].value_counts().head(10)
top_tournaments.plot(kind="bar", ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Top 10 Tournaments by Match Count")
axes[1].set_xlabel("Tournament")
axes[1].set_ylabel("Number of Matches")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

## Market Value Distribution — Top 20 Teams

In [ ]:
top20_val = mktval.nlargest(20, "squad_total_value_eur").sort_values("squad_total_value_eur")

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top20_val["team"], top20_val["squad_total_value_eur"] / 1e6, color="seagreen", edgecolor="white")
ax.set_title("Top 20 Teams by Squad Total Value")
ax.set_xlabel("Squad Total Value (€M)")
ax.set_ylabel("Team")
plt.tight_layout()
plt.show()

## Data Quality — Missing Values

In [ ]:
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print(f"{name}: no missing values")
    else:
        print(f"\n{name} — missing values:")
        print(missing.to_string())

## Team Name Alignment

In [ ]:
team_sets = {
    "fifa_rankings": set(fifa["team"].dropna()),
    "elo_ratings":   set(elo["team"].dropna()),
    "matches_home":  set(matches["home_team"].dropna()),
    "matches_away":  set(matches["away_team"].dropna()),
    "market_values": set(mktval["team"].dropna()),
    "injuries":      set(inj["team"].dropna()),
}

# Union of all match teams
match_teams = team_sets["matches_home"] | team_sets["matches_away"]
key_sets = {
    "fifa_rankings": team_sets["fifa_rankings"],
    "elo_ratings":   team_sets["elo_ratings"],
    "matches":       match_teams,
    "market_values": team_sets["market_values"],
    "injuries":      team_sets["injuries"],
}

names = list(key_sets.keys())
print(f"{'Dataset':<20} {'Teams':>6}")
print("-" * 28)
for name, s in key_sets.items():
    print(f"{name:<20} {len(s):>6}")

print("\n--- Teams in FIFA rankings NOT in matches ---")
diff = key_sets["fifa_rankings"] - key_sets["matches"]
print(sorted(diff)[:20] or "none")

print("\n--- Teams in market_values NOT in FIFA rankings ---")
diff2 = key_sets["market_values"] - key_sets["fifa_rankings"]
print(sorted(diff2)[:20] or "none")

print("\n--- Teams in injuries NOT in FIFA rankings ---")
diff3 = key_sets["injuries"] - key_sets["fifa_rankings"]
print(sorted(diff3)[:20] or "none")